In [6]:
import pandas as pd
import json
contract = {
    "columns": ["Student Name", "Class", "Semester", "Grade"],
    "allowed_classes": ["Calculus 1", "English 101", "Python 101", "Intro to Databases"],
    "allowed_semesters": ["Fall", "Spring"],
    "grade_min": 60,
    "grade_max": 100
}

print("Contract loaded:")
print(json.dumps(contract, indent=2))


Contract loaded:
{
  "columns": [
    "Student Name",
    "Class",
    "Semester",
    "Grade"
  ],
  "allowed_classes": [
    "Calculus 1",
    "English 101",
    "Python 101",
    "Intro to Databases"
  ],
  "allowed_semesters": [
    "Fall",
    "Spring"
  ],
  "grade_min": 60,
  "grade_max": 100
}


In [7]:
df = pd.read_csv("student_grades.csv")
print(f"Rows loaded: {len(df)}")
df.head()



#check colu
missing = [c for c in contract["columns"] if c not in df.columns]

if missing:
    print(f"FAIL — Missing columns: {missing}")
else:
    print("PASS — All expected columns present")

## check for nulls
null_counts = df[contract["columns"]].isnull().sum()

if null_counts.sum() == 0:
    print("PASS — No null values found")
else:
    print("FAIL — Null values found:")
    print(null_counts[null_counts > 0])




Rows loaded: 100
PASS — All expected columns present
FAIL — Null values found:
Semester    1
dtype: int64


In [9]:
## check allowed values
bad_classes = df[~df["Class"].isin(contract["allowed_classes"])]
bad_semesters = df[~df["Semester"].isin(contract["allowed_semesters"])]

if bad_classes.empty:
    print("PASS — All Class values are valid")
else:
    print(f"FAIL — {len(bad_classes)} invalid Class value(s):")
    print(bad_classes[["Student Name", "Class"]])

if bad_semesters.empty:
    print("PASS — All Semester values are valid")
else:
    print(f"FAIL — {len(bad_semesters)} invalid Semester value(s):")
    print(bad_semesters[["Student Name", "Semester"]])

FAIL — 1 invalid Class value(s):
    Student Name    Class
85  Andrew Green  Dancing
FAIL — 1 invalid Semester value(s):
          Student Name Semester
84  Stephanie Martinez      NaN


In [10]:
## cehck grade range
bad_grades = df[(df["Grade"] < contract["grade_min"]) | (df["Grade"] > contract["grade_max"])]

if bad_grades.empty:
    print(f"PASS — All grades are between {contract['grade_min']} and {contract['grade_max']}")
else:
    print(f"FAIL — {len(bad_grades)} out-of-range grade(s):")
    print(bad_grades[["Student Name", "Grade"]])

FAIL — 2 out-of-range grade(s):
      Student Name  Grade
59     Carol Perez      2
92  Dorothy Torres    105


In [11]:
checks = {
    "No missing columns":   len(missing) == 0,
    "No null values":        null_counts.sum() == 0,
    "Valid Class values":    bad_classes.empty,
    "Valid Semester values": bad_semesters.empty,
    "Grades in range":       bad_grades.empty,
}

failed_records = {
    "Valid Class values":    bad_classes,
    "Valid Semester values": bad_semesters,
    "Grades in range":       bad_grades,
}

print("\n--- Validation Summary ---")
for check, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {check}")
    if not passed and check in failed_records:
        records = failed_records[check]
        if not records.empty:
            print(records.to_string(index=False))
            print()

overall = all(checks.values())
print(f"\nOverall: {'✓ ALL CHECKS PASSED' if overall else '✗ VALIDATION FAILED'}")


--- Validation Summary ---
  [PASS] No missing columns
  [FAIL] No null values
  [FAIL] Valid Class values
Student Name   Class Semester  Grade
Andrew Green Dancing   Spring     66

  [FAIL] Valid Semester values
      Student Name      Class Semester  Grade
Stephanie Martinez Python 101      NaN     85

  [FAIL] Grades in range
  Student Name              Class Semester  Grade
   Carol Perez Intro to Databases   Spring      2
Dorothy Torres Intro to Databases     Fall    105


Overall: ✗ VALIDATION FAILED
